# B? d? li?u ?a h?nh vi

Notebook ??c b? m?i trong `data/multibehavior/`. Ch?y pipeline theo README tr??c; notebook n?y kh?ng ghi ?? d? li?u. Xem `docs/DATA_PROTOCOL.md` ?? hi?u nh?n mua 7 ng?y v? history ??ng th?i ?i?m.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.dataset import require_complete, history_for_query
DATA = require_complete(ROOT / "data/multibehavior")
summary = json.loads((ROOT / "reports/multibehavior/summary.json").read_text(encoding="utf-8"))
pd.DataFrame(summary["split_summary"]).T

## ??c query, target v? feature ri?ng

M?t query c? th? c? nhi?u target ho?c kh?ng c? target. Kh?ng ??a `target_count`, `label_end`, `has_purchase_target` ho?c `eligible_target_count` v?o feature m? h?nh.

In [ ]:
queries = pd.read_parquet(DATA / "splits/test_queries.parquet")
targets = pd.read_parquet(DATA / "splits/test_targets.parquet")
features = pd.read_parquet(DATA / "query_features.parquet")
print("Queries:", len(queries), "Target pairs:", len(targets))
print("Feature columns:", features.columns.tolist())

## V? d? l?y l?ch s? ?? bi?t

History d?ng s? ki?n <= query_time. Giao d?ch t?i query_time ?? ???c quan s?t, ch? giao d?ch sau query_time m?i c? th? l? target.

In [ ]:
events = pd.read_parquet(DATA / "events.parquet")
query = queries.iloc[0]
history = history_for_query(events, query)
assert history.event_time.le(query.query_time).all()
print("History events:", len(history))
history[["event_time", "event", "itemid", "item_idx"]].tail(10)

## Ki?m tra ch?t l??ng

?? ph? candidate kh?ng ph?i Recall/NDCG c?a m? h?nh. Query kh?ng c? mua ???c gi? trong d? li?u, nh?ng ranking metric tr?n positive queries c?n b?o c?o m?u s? ri?ng.

In [ ]:
assert all(summary["checks"].values())
pd.Series(summary["checks"], name="passed")

## B??c ti?p theo

D?ng train interactions cho Recent Popular/ItemKNN/ALS; d?ng behavior edges ho?c history cho m? h?nh ?a nhi?m. C?c m? h?nh ph?i d?ng c?ng giao th?c query/target/candidate. H?m `candidates_at_time()` trong `scripts/dataset.py` l? c?ch d?ng candidate tham chi?u, ??c l?p v?i target.